## XBRL US API - ACFR statements by report  

### Authenticate for access token 
Click the run button (Colab) or in the gray code cell below, then click the Run button above to execute the cell. Type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
	
def refresh(info):
    refresh_auth = {
                'client_id': info.client_id, 
				'client_secret' : info.client_secret, 
				'grant_type' : 'refresh_token', 
				'platform' : 'ipynb', 
				'refresh_token' : info.refresh_token 
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info	

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email, 
            'client_id': tokenInfo.client_id, 
            'client_secret' : tokenInfo.client_secret, 
            'password' : tokenInfo.password, 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s.  Run the first cell again and enter the credentials." % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
	
#print(vars(tokenInfo))
print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)

### Make a query
After the access token confirmation appears above, you can modify the query below and use the **_Cell >> Run_** menu option with the cell **immediately below this text** to run the query for updated results.

The sample results are from a set of ACFR reports posted to the XBRL US Public Filings Database.  To test for results quickly, modify the **_report\_ids_** to shorten the list, and change the **_XBRL\_Elements_** to return different data from an ACFR statement.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return.

In [ ]:
# Define the parameters for the filter and fields to be returned,

endpoint = 'cube' #taxonomy presentation linkbase + facts

# Define the parameters of the query

# query for sample ACFR reports, sort by year descending and name ascending
# https://api.xbrl.us/api/v1/report/search?report.source-name=GRIP&fields=report.entity-name,report.period-focus,report.year-focus,report.filing-date,report.id,report.entry-url,report.source-name

report_ids = [ 
# Michigan-specific sample ACFR report ids
#'737426', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/119/City-of-Flint-20220630-Annual-Accounts.xhtml	CITY OF FLINT
#'737423', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/117/City-of-Flint-20200630-Annual-Accounts.xhtml	CITY OF FLINT
#'677267', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/107/FLINTF652021.htm	Flint, Michigan
#'737425', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/118/City-of-Flint-20210630-Annual-Accounts.xhtml	City of Flint
#'737448', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/141/School-District-City-of-Flint-20210630-Annual-Accounts.xhtml	School District, City of Flint
#'737451', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/143/School-District-City-of-Flint-20230630-Annual-Accounts.xhtml	FLINT COMMUNITY SCHOOLS
#'737450', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/142/School-District-City-of-Flint-20220630-Annual-Accounts.xhtml	FLINT COMMUNITY SCHOOLS
#'737432', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/125/County-of-Ogemaw-20220930-Annual-Accounts.xhtml	County of Ogemaw
#'737431', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/124/County-of-Ogemaw-20210930-Annual-Accounts.xhtml	County of Ogemaw
#'677268', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/100/Ogemaw-20210930-Annual-Accounts.htm	County of Ogemaw
#'737430', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/123/County-of-Ogemaw-20200930-Annual-Accounts.xhtml	County of Ogemaw
#'737455', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/147/Wayne-County-20200930-Annual-Accounts.xhtml	Wayne County
#'737456', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/148/Wayne-County-20210930-Annual-Accounts.xhtml	County of Wayne
#'737457', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/149/Wayne-County-20220930-Annual-Accounts.xhtml	County of Wayne
#'737434', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/128/Davison-Community-Schools-20230630-Annual-Accounts.xhtml	Davison Community Schools (School District)
#'737433', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/127/Davison-Community-Schools-20220630-Annual-Accounts.xhtml	Davison Community Schools (School District)
#'723388', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/108/PineRiver2023.xhtml	Pine River Township

# Idaho-specific sample ACFR report ids
#'741063', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/115/City-of-Boise-20220930-Annual-Accounts.xhtml	City of Boise
#'737420', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/114/City-of-Boise-20210930-Annual-Accounts.xhtml	City of Boise
#'741061', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/113/City-of-Boise-20200930-Annual-Accounts.xhtml	City of Boise
#'737422', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/116/City-of-Coeur-DAlene-20210930-Annual-Accounts.xhtml	City of Coeur d'Alene
#'737413', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/110/City-of-Coeur-DAlene-20200930-Annual-Accounts.xhtml	City of Coeur d'Alene
#'738804', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/150/City-of-Coeur-DAlene-20230930-Annual-Accounts.xhtml	City of Coeur d'Alene

# Other sample ACFR report ids
#'677270', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/77/OAKTON2021.htm	Oakton Community College
#'737429', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/122/City-of-Provo-20230630-Annual-Accounts.xhtml	City of Provo
#'737428', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/121/City-of-Provo-20220630-Annual-Accounts.xhtml	City of Provo
#'737444', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/138/Salt-Lake-City-20220630-Annual-Accounts.xhtml	City of South Salt Lake
#'737443', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/137/Salt-Lake-City-20210630-Annual-Accounts.xhtml	City of South Salt Lake
#'677271', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/82/COD2021.htm	College of DuPage
#'737417', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/111/Jefferson-Joint-School-District-No-251-20210630-Annual-Accounts.xhtml	Jefferson Joint School District
#'737436', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/130/Jefferson-Joint-School-District-No-251-20230630-Annual-Accounts.xhtml	Jefferson Joint School District
#'737435', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/129/Jefferson-Joint-School-District-No-251-20220630-Annual-Accounts.xhtml	Jefferson Joint School District
#'737439', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/133/Nez-Perce-County-20220930-Annual-Accounts.xhtml	NEZ PERCE COUNTY
#'737438', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/132/Nez-Perce-County-20210930-Annual-Accounts.xhtml	NEZ PERCE COUNTY
#'737437', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/131/Nez-Perce-County-20200930-Annual-Accounts.xhtml	NEZ PERCE COUNTY
#'737442', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/136/North-Custer-Hospital-District-20220930-Annual-Accounts.xhtml	North Custer Hospital District
#'737441', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/135/North-Custer-Hospital-District-20210930-Annual-Accounts.xhtml	North Custer Hospital District
#'737440', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/134/North-Custer-Hospital-District-20200930-Annual-Accounts.xhtml	North Custer Hospital District
#'737427', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/120/City-of-Provo-20210630-Annual-Accounts.xhtml	Provo City
#'737445', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/139/SAN-JUAN-SCHOOL-DISTRICT-20210630-Annual-Accounts.xhtml	San Juan School District
#'737418', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/112/SAN-JUAN-SCHOOL-DISTRICT-20230630-Annual-Accounts.xhtml	SSan Juan School District
#'737446', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/140/SAN-JUAN-SCHOOL-DISTRICT-20220630-Annual-Accounts.xhtml	San Juan School District
#'737453', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/145/TriCounty-Health-Department-20221231-Annual-Accounts.xhtml	TriCounty Health Department
#'737452', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/144/TriCounty-Health-Department-20211231-Annual-Accounts.xhtml	TriCounty Health Department
#'737454', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/146/Uintah-Health-Care-Special-Service-District-20201231-Annual-Accounts.xhtml	Uintah Health Care Special Service District
#'677269', #https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/106/HARPER2021.htm	William Rainey Harper College
]

# query for unique Statements in the GRIP Taxonomies
# login at https://xbrl.us/wp-login.php?redirect_to=/xbrl-api-community then paste the query below into the browser to see the list
# https://api.xbrl.us/api/v1/relationship/search?dts.id=821819,926240&network.link-name=presentationLink&fields=network.role-description&unique

Statements = [
		# remove the Statement between the quotes to get all data for report.ids below
		'200110 - Statement - Activities - General Revenues and Changes in Net Position'
		] 

# Define data fields to return (multi-sort based on order)

fields = [ # this is the list of the characteristics of the data being returned by the query
		'report.id',
		'dts.id',
		'cube.description.sort(ASC)',
		'cube.tree-sequence.sort(ASC)',
		'report.entity-name',
		'dimension-pair',
		'cube.primary-local-name',
		'fact.value',
		'unit',
		'period.fiscal-year.sort(DESC)',
		'period.fiscal-period'
        ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 10 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.
 
params = {
         'report.source-name': 'GRIP', 
         'cube.description': ','.join(Statements),
         #'report.id': ','.join(report_ids), #comment out to get all reports
         'fields': ','.join(fields),
         'unique': ''
         }

print('\n\nNext click the run button (Colab) or in the gray code cell below, then click the Run button above to execute the query for results.\n\n')


In [ ]:
# @title
### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Mon Jun 30 13:36:35 2025 training@xbrl.us (client ID: 69e1257c ...) started the query and
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 1306 records.

At Mon Jun 30 13:36:35 2025, the query finished with   1306   rows returned in -1 day, 23:59:59.712217 for 
https://api.xbrl.us/api/v1/cube/search?unique&report.source-name=GRIP&cube.description=200110+-+Statement+-+Activities+-+General+Revenues+and+Changes+in+Net+Position&fields=report.id,dts.id,cube.description.sort(ASC),cube.tree-sequence.sort(ASC),report.entity-name,dimension-pair,cube.primary-local-name,fact.value,unit,period.fiscal-year.sort(DESC),period.fiscal-period&unique=


,report.id,dts.id,cube.description,cube.tree-sequence,report.entity-name,dimension-pair,cube.primary-local-name,fact.value,unit,period.fiscal-year,period.fiscal-period
0,723388,926240,200110 - Statement - Activities - General Revenues and Changes in Net Position,1,Pine River Township,[{'TypeOfGovernmentUnitAxis': 'BusinessTypeActivitiesMember'}],NetExpenseRevenue,57711,USD,2023,2022-04-01 - 2023-04-01
1,723388,926240,200110 - Statement - Activities - General Revenues and Changes in Net Position,1,Pine River Township,[{'TypeOfGovernmentUnitAxis': 'GovernmentalActivitiesMember'}],NetExpenseRevenue,528371,USD,2023,2022-04-01 - 2023-04-01
2,723388,926240,200110 - Statement - Activities - General Revenues and Changes in Net Position,1,Pine River Township,[{'TypeOfGovernmentUnitAxis': 'PrimaryGovernmentActivitiesMember'}],NetExpenseRevenue,470660,USD,2023,2022-04-01 - 2023-04-01
3,737418,926240,200110 - Statement - Activities - General Revenues and Changes in Net Position,1,"SAN JUAN SCHO<span class=""_ _0""></span>OL DISTRI<span class=""_ _0""></span>CT",[{'TypeOfGovernmentUnitAxis': 'BusinessTypeActivitiesMember'}],NetExpenseRevenue,-155037,USD,2023,2022-07-01 - 2023-07-01
4,737418,926240,200110 - Statement - Activities - General Revenues and Changes in Net Position,1,"SAN JUAN SCHO<span class=""_ _0""></span>OL DISTRI<span class=""_ _0""></span>CT",[{'TypeOfGovernmentUnitAxis': 'GovernmentalActivitiesMember'}],NetExpenseRevenue,-18292891,USD,2023,2022-07-01 - 2023-07-01
...,...,...,...,...,...,...,...,...,...,...,...
1301,737455,926240,200110 - Statement - Activities - General Revenues and Changes in Net Position,145,"Wayne<span class=""_ _75""> </span>County",[{'TypeOfGovernmentUnitAxis': 'GovernmentalActivitiesMember'}],NetPositionAtBeginningOfPeriodAfterAdjustments,687957319,USD,2020,3Q
1302,737455,926240,200110 - Statement - Activities - General Revenues and Changes in Net Position,145,"Wayne<span class=""_ _75""> </span>County",[{'TypeOfGovernmentUnitAxis': 'PrimaryGovernmentActivitiesMember'}],NetPositionAtBeginningOfPeriodAfterAdjustments,770436136,USD,2020,3Q
1303,741061,926240,200110 - Statement - Activities - General Revenues and Changes in Net Position,145,City of Boise,[{'TypeOfGovernmentUnitAxis': 'BusinessTypeActivitiesMember'}],NetPositionAtBeginningOfPeriodAfterAdjustments,638992000,USD,2020,3Q
1304,741061,926240,200110 - Statement - Activities - General Revenues and Changes in Net Position,145,City of Boise,[{'TypeOfGovernmentUnitAxis': 'GovernmentalActivitiesMember'}],NetPositionAtBeginningOfPeriodAfterAdjustments,375638000,USD,2020,3Q


In [ ]:
# If you run this program locally, you can save the output to a file on your computer (modify D:\results.csv to your system)
df.to_csv(r"D:\results.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('data.csv')
#!cp data.csv "drive/My Drive/"